In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from utilsforecast.feature_engineering import future_exog_to_historic

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import GMM, MQLoss, DistributionLoss
from neuralforecast.losses.pytorch import MAE
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('./ferreteria.csv')
df

In [ ]:
df['ds'] = pd.to_datetime(df['ds'])

In [ ]:
df.dtypes

In [ ]:
# Tendencia demanda total
df_agg = df.groupby('ds')['demand_units'].sum().reset_index()
plt.figure(figsize=(14,6))
plt.plot(df_agg['ds'], df_agg['demand_units'])
plt.title('Demanda Total Semanal (Todos SKUs Ferretería)')
plt.ylabel('Demanda (unidades)')
plt.show()

In [ ]:
# Correlación variables principales con demanda
main_vars = ['demand_units', 'price_unit', 'promotion_active', 'season_factor', 'temperature_avg', 'events_nearby', 'marketing_spend', 'competitor_price', 'economic_index', 'construction_index', 'online_sales_rate', 'store_visits']
corr = df[main_vars].corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlación Variables Principales con Demanda')
plt.show()

In [ ]:
# Distribución demanda por SKU
plt.figure(figsize=(16,8))
sns.boxplot(x='sku_id', y='demand_units', data=df)
plt.title('Distribución Demanda por SKU')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Demanda vs promoción activa
plt.figure(figsize=(10,6))
sns.violinplot(x='promotion_active', y='demand_units', data=df)
plt.title('Demanda con y sin Promoción')
plt.show()

In [ ]:
# Demanda vs promoción activa
plt.figure(figsize=(10,6))
sns.boxplot(x='promotion_active', y='demand_units', data=df)
plt.title('Demanda con y sin Promoción')
plt.show()

In [ ]:
# Demanda vs índice construcción
plt.figure(figsize=(10,6))
sns.scatterplot(x='construction_index', y='demand_units', hue='sku_id', data=df.sample(1000), alpha=0.6)
plt.title('Demanda vs Índice Construcción (muestra)')
plt.show()

In [ ]:
# =======================================================
# 3. Preparación y Modelo MLForecast con CatBoost
# =======================================================
df_ts = df[['ds','sku_id', 'demand_units', 'price_unit', 'promotion_active', 'season_factor', 'temperature_avg', 'events_nearby', 'marketing_spend', 
'competitor_price', 'economic_index', 'online_sales_rate', 'store_visits', 'holiday_week', 'lead_time_days', 'construction_index']
]

df_ts = df_ts.rename(columns={'sku_id': 'unique_id', 'demand_units': 'y'})

exog_cols = ['price_unit', 'promotion_active', 'season_factor', 'temperature_avg', 'events_nearby', 'marketing_spend', 
'competitor_price', 'economic_index', 'online_sales_rate', 'store_visits', 'holiday_week', 'lead_time_days', 'construction_index']

In [ ]:
df['unique_id'] = df['sku_id']

df_ts = df[['unique_id', 'ds', 'demand_units', 'price_unit', 'promotion_active', 'season_factor', 'temperature_avg', 'events_nearby', 'marketing_spend',
             'competitor_price', 'economic_index', 'online_sales_rate', 'store_visits', 'holiday_week', 'lead_time_days', 'construction_index']].copy()
df_ts = df_ts.rename(columns={'demand_units': 'y'})

exog_cols = ['price_unit', 'promotion_active', 'season_factor', 'temperature_avg', 'events_nearby', 'marketing_spend', 'competitor_price', 
             'economic_index', 'online_sales_rate', 'store_visits', 'holiday_week', 'lead_time_days', 'construction_index']



df_ts.head()

In [ ]:
train_df = df_ts[df_ts['ds'] < '2024-01-01']
test_df = df_ts[df_ts['ds'] >= '2024-01-01']

train_df.shape, test_df.shape

In [ ]:
horizon_base = 52

models_nf = [NHITS(h= horizon_base, 
                   input_size= 52 *2,  
                       loss = DistributionLoss(distribution='Normal', level=[80, 90], return_params=True), 
                       valid_loss =  MQLoss(level = [80, 95]),
                       scaler_type = 'robust',
                       max_steps=500, 
                       n_freq_downsample=[4, 2, 1],
                       hist_exog_list = exog_cols,
                       futr_exog_list=exog_cols,
                       )]

In [ ]:
nf_base = NeuralForecast(models=models_nf, freq='W-MON')

In [ ]:
nf_base.fit(df=train_df)

In [ ]:
_, futr_base = future_exog_to_historic(df=train_df, freq= 'W-MON', features=exog_cols, h=horizon_base)

In [ ]:
forecast_base = nf_base.predict(futr_df=futr_base)
forecast_base

In [ ]:
forecast_base.columns

In [ ]:
from utilsforecast.plotting import plot_series
plot_series(train_df, forecast_base, models= ['NHITS', 'NHITS-median',])

In [ ]:
plot_series(train_df, forecast_base, models= ['NHITS', ], level= [90])

In [ ]:
def sensitivity_analysis(variable, multipliers):
    results = []
    for mult in multipliers:
        _, futr_sens = future_exog_to_historic(df = train_df, freq='W-MON', features=exog_cols, h=52)
        futr_sens[variable] *= mult

        # realizar forecast
        forecast_sens = nf_base.predict(futr_df=futr_sens)
        mean_demand = forecast_sens.groupby('unique_id')['NHITS'].mean().mean()
        results.append(mean_demand)
        
    return results

In [ ]:
import pandas as pd

def sensitivity_analysis(variable, multipliers):
    results = []

    for mult in multipliers:
        # Generar futuros exógenos
        _, futr_sens = future_exog_to_historic(
            df=train_df,
            freq='W-MON',
            features=exog_cols,
            h=52
        )

        # Convertir la columna a float para permitir multiplicadores decimales
        futr_sens[variable] = futr_sens[variable].astype(float)

        # Aplicar multiplicador por SKU
        for sku in futr_sens['unique_id'].unique():
            futr_sens.loc[futr_sens['unique_id'] == sku, variable] *= mult

        # Forecast
        forecast_sens = nf_base.predict(futr_df=futr_sens)

        # Media de demanda por SKU
        mean_demand = forecast_sens.groupby('unique_id')['NHITS'].mean().reset_index()
        mean_demand['multiplier'] = mult
        mean_demand['variable'] = variable

        results.append(mean_demand)

    # Concatenar resultados
    results_df = pd.concat(results, ignore_index=True)
    results_df = pd.DataFrame(results_df)

    pronostico = pd.DataFrame(forecast_sens)

    return results_df, pronostico


In [ ]:
print("\nSensibilidad a Promoción:")
promo_sensi = sensitivity_analysis('promotion_active', [0.0, 1.0, 1.5])
promo_sensi

In [ ]:
promo_sensi[1]

In [ ]:
promo_sensi.groupby('unique_id')['NHITS'].mean().reset_index()

In [87]:
import pandas as pd

def sensitivity_analysis(variable, multipliers):
    results = []

    for mult in multipliers:
        # Generar futuros exógenos
        _, futr_sens = future_exog_to_historic(
            df=train_df,
            freq='W-MON',
            features=exog_cols,
            h=52
        )

        # Convertir a float para evitar errores de dtype
        futr_sens[variable] = futr_sens[variable].astype(float)

        # Aplicar multiplicador por SKU
        for sku in futr_sens['unique_id'].unique():
            futr_sens.loc[futr_sens['unique_id'] == sku, variable] *= mult

        # Forecast completo
        forecast_sens = nf_base.predict(futr_df=futr_sens)

        # Renombrar la columna NHITS para identificar el multiplicador
        forecast_sens = forecast_sens[['unique_id', 'ds', 'NHITS']].copy()
        forecast_sens.rename(columns={'NHITS': f'NHITS_mult_{mult}'}, inplace=True)

        results.append(forecast_sens)

    # Unir todos los forecasts por SKU y fecha
    results_df = results[0]
    for df in results[1:]:
        results_df = results_df.merge(df, on=['unique_id', 'ds'], how='left')

    return results_df


In [89]:
promo_sensi = sensitivity_analysis('promotion_active', [0.0, 1.0, 1.5])

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

In [90]:
promo_sensi

,unique_id,ds,NHITS_mult_0.0,NHITS_mult_1.0,NHITS_mult_1.5
0,"Brochas 2""",2024-01-01,1065.006348,1063.895264,1052.791504
1,"Brochas 2""",2024-01-08,859.514648,854.663513,851.815430
2,"Brochas 2""",2024-01-15,1034.458252,1065.934082,1082.032593
3,"Brochas 2""",2024-01-22,1380.643311,1398.592407,1405.236816
4,"Brochas 2""",2024-01-29,1337.368286,1344.941284,1344.325195
...,...,...,...,...,...
775,"Tubos PVC 1/2""",2024-11-25,429.584198,397.868195,391.490631
776,"Tubos PVC 1/2""",2024-12-02,661.402283,612.935425,580.730835
777,"Tubos PVC 1/2""",2024-12-09,352.237335,368.398895,400.224762
778,"Tubos PVC 1/2""",2024-12-16,577.990906,614.502502,643.491638
